In [ ]:
import os
import requests, time, random, hashlib, json
from datetime import datetime, timezone, timedelta
import csv

# Set Header
API_URL = "https://open-kr.aqara.com/v3.0/open/api"

# 비밀값(APPID/KEYID/APPKEY/ACCESS_TOKEN)은 환경변수에서만 로드한다.
# CLAUDE.md §2.5 / §6 — 하드코딩 금지. Colab 사용 시 다음 셀을 먼저 실행해 주입:
#   import os
#   os.environ["AQARA_APPID"] = "..."        # Aqara 콘솔 발급값
#   os.environ["AQARA_KEYID"] = "..."
#   os.environ["AQARA_APPKEY"] = "..."
#   os.environ["AQARA_ACCESS_TOKEN"] = "..." # 약 7일 간격 갱신
def _require_env(name: str) -> str:
    val = os.environ.get(name)
    if not val:
        raise RuntimeError(
            f"환경변수 {name} 가 설정되어 있지 않습니다. 노트북 상단 셀에서 os.environ 으로 주입 후 재실행하세요."
        )
    return val

APPID = _require_env("AQARA_APPID")
KEYID = _require_env("AQARA_KEYID")
APPKEY = _require_env("AQARA_APPKEY")
# Access Token 찾는 법 (약 7일 간격 갱신):
#   https://developer.aqara.com/ 로그인 → Console > Manage Project > Detail
#   > Authorization management > Authorization details
ACCESS_TOKEN = _require_env("AQARA_ACCESS_TOKEN")

# Utility 함수
def kst_to_utc_millis(kst_str):
    """
    입력: 'YYYY-MM-DD HH:MM:SS' (KST)
    출력: UTC 기준 millisecond timestamp (문자열)
    """
    kst = timezone(timedelta(hours=9))
    dt = datetime.strptime(kst_str, "%Y-%m-%d %H:%M:%S")
    dt = dt.replace(tzinfo=kst)
    utc_dt = dt.astimezone(timezone.utc)
    return str(int(utc_dt.timestamp() * 1000))

def normalize_subject_id(subject_id: str) -> str:
    """
    입력: 4CF8CDF3C829ECD
    출력: lumi.4cf8cdf3c829ecd
    이미 lumi.가 붙어있으면 소문자만 정리
    """
    subject_id = subject_id.strip()

    if subject_id.lower().startswith("lumi."):
        return subject_id.lower()

    return "lumi." + subject_id.lower()

# Request data
def get_history_of_device_attr(subjectId, resourceIds, start_kst, end_kst):
    start_ms = kst_to_utc_millis(start_kst)
    end_ms   = kst_to_utc_millis(end_kst)
    print(start_ms, end_ms)
    print(normalize_subject_id(subjectId))

    nonce = str(random.randint(100000, 999999))
    ts = str(int(time.time() * 1000))
    pre_str = f"Accesstoken={ACCESS_TOKEN}&Appid={APPID}&Keyid={KEYID}&Nonce={nonce}&Time={ts}{APPKEY}"
    sign = hashlib.md5(pre_str.lower().encode("utf-8")).hexdigest()

    header = {
        "Appid": APPID,
        "Keyid": KEYID,
        "Accesstoken": ACCESS_TOKEN,
        "Nonce": nonce,
        "Time": ts,
        "Sign": sign,
        "Content-Type": "application/json"
    }

    payload = {
        "intent": "fetch.resource.history",
        "data": {
        "subjectId": normalize_subject_id(subjectId),
        "resourceIds": [resourceIds],
        "startTime": start_ms,
        "endTime": end_ms
        }
    }
    resp = requests.post(API_URL, headers=header, json=payload)
    return resp

# get_list_of_kst_and_value
def get_list_of_kst_and_value(resp):
    result = resp.json()
    if result.get("code") != 0:
        print("API Error:", result)
        return []
    data_list = result.get("result", {}).get("data", [])
    kst_and_value_list = []
    for item in data_list:
        ts = int(item["timeStamp"])
        value = item["value"]
        # UTC → KST 변환
        utc_dt = datetime.fromtimestamp(ts / 1000, tz=timezone.utc)
        kst_dt = utc_dt.astimezone(timezone(timedelta(hours=9)))
        kst_and_value_list.append((
            kst_dt.strftime('%Y-%m-%d %H:%M:%S'),
            value
        ))
    return kst_and_value_list


# 다운 받고 싶은 데이터 정보
# 다운 받고 싶은 기간 (한 번에 최대 100개의 데이터 나옴, 7일 이내)
start_kst = "2026-02-13 00:00:00"
end_kst   = "2026-02-23 23:59:00"
# 아카라 앱 상 장치ID 또는 lumi.로 시작하는 형태
subject_id = "4CF8CDF3C829AED"
# 진동센서 모션감지 속성 (1 detected, 255는 deactivated로 변경될 때)
resource_id = "13.7.85" #Tap은 13.3.85

total_data_list = []
resp = get_history_of_device_attr(subject_id, resource_id, start_kst, end_kst)
kst_and_value_list = get_list_of_kst_and_value(resp)
print(len(kst_and_value_list))
total_data_list += kst_and_value_list

while len(kst_and_value_list) > 99:
    end_kst = kst_and_value_list[-1][0]
    print(end_kst)
    dt = datetime.strptime(end_kst, "%Y-%m-%d %H:%M:%S")
    dt_plus_1 = dt + timedelta(seconds=1)
    end_kst = dt_plus_1.strftime("%Y-%m-%d %H:%M:%S")

    resp = get_history_of_device_attr(subject_id, resource_id, start_kst, end_kst)
    kst_and_value_list = get_list_of_kst_and_value(resp)
    print(len(kst_and_value_list))
    total_data_list += kst_and_value_list

with open('Output_WS9700_Middle.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['time', 'value'])  # header
    writer.writerows(total_data_list)
print(len(total_data_list))